In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import psycopg2

# accesing the data from post gre sql
conn=psycopg2.connect(
    host='localhost',
    database='onlineretail',
    user='postgres',
    password='1234',
    port='5433'
)
df=pd.read_sql('SELECT * FROM orders;',conn)
cursor=conn.cursor()
df.head()



In [ ]:
# changing data type of date and time column    
df['invoicedate']=df['invoicedate'].astype(str)

df['invoice_date']=df['invoicedate'].str.split(' ').str[0]

df['invoice_time']=df['invoicedate'].str.split(' ').str[1]

df['invoice_time'] = pd.to_datetime(df['invoice_time'], format='%H:%M:%S', errors='coerce')
df['invoice_date']=pd.to_datetime(df['invoice_date'])

df=df.drop(columns=['invoicedate'])

# c
df['customerid']=df['customerid'].astype(str)
df['customerid']=df['customerid'].str.split('.').str[0]

# filling null values into 0 for customerid 

    
# df['customerid']=df['customerid'].astype(int)
df.dropna(subset=['customerid'],inplace=True)

df['customerid'].isnull().sum()

 
df['invoiceno'].duplicated().sum()

In [ ]:
# removing duplicate invoices in dataset
df=df.drop_duplicates()


df.duplicated().sum()


In [ ]:
# removing unit when it 0

# THERE ARE 2519 VALUES ARE PRESENT WHEN UNIT PRICE IS 0

cursor.execute("""DELETE FROM orders WHERE unitprice=0;""")

In [ ]:
# STANDARDIZE COUNTRY NAMES
country_map = {
    'United Kingdom': 'UK',
    'France': 'FR',
    'Germany': 'DE',
    'Switzerland': 'CH',
    'EIRE': 'IE',        
    'Spain': 'ES',
    'Netherlands': 'NL',
    'Belgium': 'BE',
    'Portugal': 'PT',
    'Australia': 'AU',
    'Norway': 'NO',
    'Italy': 'IT',
    'Channel Islands': 'CIN',
    'Finland': 'FI',
    'Cyprus': 'CY',
    'Sweden': 'SE',
    'Austria': 'AT',
    'Denmark': 'DK',
    'Japan': 'JP',
    'Poland': 'PL',
    'USA': 'US',
    'Israel': 'IL',
    'Unspecified': 'UNSP',
    'Singapore': 'SG',
    'Iceland': 'IS',
    'Canada': 'CA',
    'Greece': 'GR',
    'Malta': 'MT',
    'United Arab Emirates': 'AE',
    'European Community': 'EU',
    'RSA': 'ZA',
    'Lebanon': 'LB',
    'Lithuania': 'LT',
    'Brazil': 'BR',
    'Czech Republic': 'CZ',
    'Bahrain': 'BH',
    'Saudi Arabia': 'SA'
}

df['country'] = df['country'].replace(country_map)
df

In [ ]:
# making revenue column 
conn.rollback()


# query=pd.read_sql('SELECT * FROM orders;',conn)
# query.head()

cursor.execute("""UPDATE orders SET total_revenue=(quantity*unitprice);""")

conn.commit()

In [ ]:
# making orders per customer column 

df['order_per_customer']=df.groupby('customerid')['invoiceno'].transform('count')

In [ ]:
# AVERAGE ORDER VALUE 
df['average_order_value']=df.groupby('invoiceno')['total_revenue'].transform('mean')


df['average_order_value']=df['average_order_value'].astype(str)

df['average_order_value']=df['average_order_value'].str.split('.').str[0]

df['average_order_value']=df['average_order_value'].astype(int)
df.info()

In [ ]:
# return rate per product
df['is_returns']=df['status'].apply(lambda x: 1 if x=='Cancelled' else 0  )

df_unique=df.dropna(subset=['invoiceno','description'])

df_unique = df_unique.drop_duplicates(subset=['invoiceno', 'description'])

return_rate_per_product=df_unique.groupby('description')['is_returns'].mean()*100


print(return_rate_per_product)

In [ ]:
# MONTHLY REVENUE TREND 

df['invoice_date']=df['invoice_date'].astype(str)
df['invoice_month']=df['invoice_date'].str.split('-').str[1]


monthly_revenue=df.groupby('invoice_month')['total_revenue'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.set_style("whitegrid")

sns.lineplot(
    data=monthly_revenue,
    x="invoice_month",
    y="total_revenue",
    marker="o",           
    linewidth=2.5,
    markersize=7
)

plt.title("Monthly Revenue Trend", fontsize=14)
plt.xlabel("invoice_month")
plt.ylabel("total_revenue")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 📊 Monthly Revenue Insight

> **Note:** Insights are based on revenue data only. Further analysis with additional data points 
> (e.g. marketing spend, product category) would provide deeper insights.

---

## Phase 1 — Stable Period (Jan – Aug)

- Revenue remained **stable between ₹4.25L – ₹6.15L** throughout the first 8 months
- No significant growth observed, likely due to **absence of major festivals** during this period
- **April recorded the lowest revenue (₹4.25L)**, likely due to **post-financial year reset** and no major festivals

---

## Phase 2 — Growth Phase (Sep – Nov)

- Revenue **jumped sharply from ₹9.29L to ₹11.27L**
- Likely driven by **major festive seasons** — Navratri, Dussehra, and Diwali
- November was the **peak month of the year at ₹11.27L**

---

## Phase 3 — Decline (Dec)

- Revenue **declined by 20.7%** (₹11.27L → ₹8.94L)
- Likely due to **post-festive slowdown** as consumer spending normalized after the festive season

---

## 📌 Key Takeaways

| Month | Revenue | Highlight |
|-------|---------|-----------|
| April | ₹4.25L | Lowest month of the year |
| Aug → Sep | ₹6.15L → ₹9.29L | Biggest single jump (+51%) |
| November | ₹11.27L | Peak of the year |
| December | ₹8.94L | Post-festive decline (-20.7%) |

In [ ]:
print(df['total_revenue'].describe())
print(df['total_revenue'].sum())

In [ ]:
# TOP 10 HIGH REVENUE PRODUCTS

products_revenue=df.groupby('description')['total_revenue'].sum().nlargest(10)

products_revenue.plot(kind='bar', figsize=(10, 6))
plt.title('Top 10 products')
plt.xlabel('Product Description')
plt.ylabel('Total Revenue')
plt.show()

products_revenue

## Top 10 Products — Revenue Insights

---

### Product Revenue Table

| Rank | Product | Revenue (GBP) |
|------|---------|--------------|
| 1 | Regency Cakestand 3 Tier | 1,32,567.70 |
| 2 | White Hanging Heart T-Light Holder | 93,767.80 |
| 3 | Jumbo Bag Red Retrospot | 83,056.52 |
| 4 | Party Bunting | 67,628.43 |
| 5 | Postage ⚠️ | 66,710.24 |
| 6 | Assorted Colour Bird Ornament | 56,331.91 |
| 7 | Rabbit Night Light | 51,042.84 |
| 8 | Chilli Lights | 45,915.41 |
| 9 | Paper Chain Kit 50's Christmas | 41,423.78 |
| 10 | Picnic Basket Wicker 60 Pieces | 39,619.50 |

---

In [ ]:

# TOP 10 RETURNED PRODUCTS

returned = df[df['status'] == 'Cancelled']

return_products = returned.groupby('description')['total_revenue'].sum().abs().nlargest(10)

return_products.plot(kind='bar', figsize=(10,6))
plt.title('Top 10 Returned Products by Revenue Lost')
plt.xlabel('Product Description')
plt.ylabel('Revenue Lost (GBP £)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print(return_products)


## Top 10 Returned Products — Revenue Lost Insights

---

### Returned Products Revenue Loss Table

| Rank | Product | Revenue Lost (GBP) |
|------|---------|----------------------|
| 1 | Paper Craft, Little Birdie | 1,68,469.60 |
| 2 | Manual ⚠️ | 1,10,000+ |
| 3 | Medium Ceramic Top Storage Jar | 75,000+ |
| 4 | Postage ⚠️ | 10,000+ |
| 5 | Regency Cakestand 3 Tier | 8,500+ |
| 6 | CRUK Commission ⚠️ | 7,000+ |
| 7 | White Hanging Heart T-Light Holder | 6,000+ |
| 8 | Fairy Cake Flannel Assorted Colour | 5,500+ |
| 9 | Discount ⚠️ | 5,000+ |
| 10 | Pantry Chopping Board | 4,500+ |


In [ ]:
# top 20% customers , how much revenue?

# non cancelled filtering
clean_df=df[df['status']!='Cancelled']

# total revenue by customer id 

customer_revenue=clean_df.groupby('customerid')['total_revenue'].sum().sort_values(ascending=False)

total_customers=len(customer_revenue)

top_20_count=int(total_customers*0.20)

top_20_customers=customer_revenue.head(top_20_count)



top_20_revenue = top_20_customers.sum()
total_revenue = customer_revenue.sum()
percentage = (top_20_revenue / total_revenue) * 100


print(f"Top 20% Customers: {top_20_count}")
print(f"Top 20% Revenue: {top_20_revenue:,.2f}")


labels = [f'Top 20% Customers\n({top_20_count})', 
          f'Bottom 80% Customers\n({total_customers - top_20_count})']

values = [top_20_revenue, total_revenue - top_20_revenue]


plt.bar(labels, values, color=['#2196F3', '#E0E0E0'])
plt.title('Top 20% Customers — Revenue Contribution')
plt.ylabel('Total Revenue (GBP)')
plt.tight_layout()
plt.show()

## Top 20% Customers — Revenue Insight

| Segment | Customers | Revenue (GBP) |
|---------|-----------|---------------|
| Top 20% | 867 | 6,600,000+ |
| Bottom 80% | 3,472 | 2,200,000+ |

- Top 20% customers (867) generate nearly **75% of total revenue**
- Bottom 80% customers (3,472) contribute only **25% of total revenue**
- This follows the **Pareto Principle** — majority of revenue comes from a small group of customers
- Retaining these 867 high-value customers is critical for business stability

In [ ]:
# which country gives highest revenue 


total_revenue_by_country=df.groupby('country')['total_revenue'].sum().nlargest(10)

total_revenue_by_country.plot(kind='bar')
plt.title('total revenue by country')
plt.xlabel('country names')
plt.ylabel('revenue')

plt.show()
total_revenue_by_country

## Top 10 Countries — Revenue Insights

| Rank | Country | Revenue (GBP) |
|------|---------|---------------|
| 1 | United Kingdom | 67,47,156.15 |
| 2 | Netherlands | 2,84,661.54 |
| 3 | Ireland | 2,50,001.78 |
| 4 | Germany | 2,21,509.47 |
| 5 | France | 1,96,626.05 |
| 6 | Australia | 1,37,009.77 |
| 7 | Switzerland | 55,739.40 |
| 8 | Spain | 54,756.03 |
| 9 | Belgium | 40,910.96 |
| 10 | Sweden | 36,585.41 |

- **UK dominates** — generates 67,47,156 revenue, significantly higher than all other countries combined
- **24x gap** — 2nd ranked Netherlands (2,84,661) is nearly 24x less than UK
- **Australia stands out** — only non-European country in top 10
- **Bottom 3** (Switzerland, Belgium, Sweden) all below 60,000 — very low international presence


## 💡 3 Recommendations

1. **Increase festive season inventory** — Since Sep–Nov drives maximum 
   revenue, stock and marketing should be increased before festive season

2. **Retain top 20% customers** — Introduce loyalty programs or 
   special offers for 867 high-value customers to maintain revenue stability

3. **Investigate high return products** — Paper Craft Little Birdie 
   has unusually high returns — quality or product description

should be reviewed to reduce revenue loss

In [ ]:
print("started")
df.to_excel(
    "clean_orders.xlsx",
    index=False,
)

print("Excel file created successfully!")